# Labs C & D — Denoising and the interpolation walk

Companion to *Latent Space*, Chapters 5 and 6.

**Runtime → Change runtime type → T4 GPU.** Required, not optional.

Lab D is the most convincing artifact in the book. If you're short on time,
skip to it.


In [ ]:
!nvidia-smi -L
!pip -q install diffusers transformers accelerate safetensors

In [ ]:
# Writes this lab's files into Colab. Nothing to download, nothing
# to clone, nothing to upload. Just run this cell.
import base64, pathlib

FILES = {
    "denoise_in_public.py": (
        "IiIiCkxhYiBDOiB3YXRjaCB0aGUgd2Fsay4KCkdlbmVyYXRpb24gaXMgbm90IHJldHJpZXZhbCBhbmQgaXQgaXMgbm90IG9uZS1zaG90IGNvbnN0cnVjdGlv"
        "bi4gSXQgaXMgYSB3YWxrCnRoYXQgc3RhcnRzIGF0IHJhbmRvbSBub2lzZSBhbmQgc3RlcHMgdG93YXJkIHRoZSByZWdpb24gb2YgbGF0ZW50IHNwYWNlIHdo"
        "ZXJlCnJlYWwgaW1hZ2VzIGxpdmUuIFRoaXMgc2NyaXB0IHNhdmVzIGV2ZXJ5IHN0ZXAgc28geW91IGNhbiBsb29rIGF0IHRoZSB3YWxrLgoKUnVuOiAgcHl0"
        "aG9uIGRlbm9pc2VfaW5fcHVibGljLnB5CiAgICAgIHB5dGhvbiBkZW5vaXNlX2luX3B1YmxpYy5weSAtLXByb21wdCAiYSBsaWdodGhvdXNlIGluIGEgc3Rv"
        "cm0iIC0tc3RlcHMgMzAKICAgICAgcHl0aG9uIGRlbm9pc2VfaW5fcHVibGljLnB5IC0tc3dlZXAtc3RlcHMgICAgICAjIHF1YWxpdHkvY29zdCBmcm9udGll"
        "cgogICAgICBweXRob24gZGVub2lzZV9pbl9wdWJsaWMucHkgLS1zd2VlcC1ndWlkYW5jZSAgICMgd2hhdCBDRkcgYWN0dWFsbHkgZG9lcwogICAgICBweXRo"
        "b24gZGVub2lzZV9pbl9wdWJsaWMucHkgLS1zd2VlcC1zZWVkcyAgICAgICMgc2FtZSB3b3JkcywgZGlmZmVyZW50IHN0YXJ0CgpPdXRwdXRzIGxhbmQgaW4g"
        "Li9vdXRwdXQvLiBMb29rIGF0IHRoZW0gaW4gb3JkZXIuCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBvcwppbXBvcnQgc3lzCgpERUZBVUxUX1BST01Q"
        "VCA9ICJhIHN0b25lIGNhc3RsZSBvbiBhIGNsaWZmIGluIGhlYXZ5IGZvZywgZHJhbWF0aWMgbGlnaHQiCgoKZGVmIGxvYWRfcGlwZWxpbmUobW9kZWxfaWQs"
        "IGRldmljZV9wcmVmPU5vbmUpOgogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBTdGFibGVEaWZm"
        "dXNpb25QaXBlbGluZQogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHN5cy5leGl0KAogICAgICAgICAgICAiTWlzc2luZyBkZXBlbmRlbmNpZXMu"
        "IFJ1bjpcbiIKICAgICAgICAgICAgIiAgcGlwIGluc3RhbGwgLXIgcmVxdWlyZW1lbnRzLnR4dFxuIgogICAgICAgICAgICAiU2VlIC4uL1NFVFVQLm1kIGZv"
        "ciB0aGUgcmVudGVkLUdQVSByb3V0ZSBpZiB5b3UgaGF2ZSBubyBHUFUuIgogICAgICAgICkKCiAgICAjIHBpY2sgYSBkZXZpY2UKICAgIGlmIGRldmljZV9w"
        "cmVmOgogICAgICAgIGRldmljZSA9IGRldmljZV9wcmVmCiAgICBlbGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgZGV2aWNlID0gImN1"
        "ZGEiCiAgICBlbGlmIGdldGF0dHIodG9yY2guYmFja2VuZHMsICJtcHMiLCBOb25lKSBhbmQgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpOgog"
        "ICAgICAgIGRldmljZSA9ICJtcHMiCiAgICBlbHNlOgogICAgICAgIGRldmljZSA9ICJjcHUiCgogICAgZHR5cGUgPSB0b3JjaC5mbG9hdDE2IGlmIGRldmlj"
        "ZSA9PSAiY3VkYSIgZWxzZSB0b3JjaC5mbG9hdDMyCiAgICBpZiBkZXZpY2UgPT0gImNwdSI6CiAgICAgICAgcHJpbnQoIldBUk5JTkc6IG5vIEdQVSBmb3Vu"
        "ZC4gVGhpcyB3aWxsIHJ1biBvbiBDUFUgYW5kIGJlIHZlcnkgc2xvdyIpCiAgICAgICAgcHJpbnQoIiAgICAgICAgIChzZXZlcmFsIG1pbnV0ZXMgcGVyIGlt"
        "YWdlKS4gQ29uc2lkZXIgLS1zdGVwcyA4LCBvciIpCiAgICAgICAgcHJpbnQoIiAgICAgICAgIHNlZSAuLi9TRVRVUC5tZCBmb3IgcmVudGluZyBhIEdQVSBm"
        "b3IgYW4gaG91ci5cbiIpCgogICAgcHJpbnQoZiJMb2FkaW5nIHttb2RlbF9pZH0gb24ge2RldmljZX0gLi4uIikKICAgIHByaW50KCIoRmlyc3QgcnVuIGRv"
        "d25sb2FkcyB+NEdCLiBPbmNlIG9ubHkuKVxuIikKCiAgICBwaXBlID0gU3RhYmxlRGlmZnVzaW9uUGlwZWxpbmUuZnJvbV9wcmV0cmFpbmVkKAogICAgICAg"
        "IG1vZGVsX2lkLCB0b3JjaF9kdHlwZT1kdHlwZSwgc2FmZXR5X2NoZWNrZXI9Tm9uZSwgcmVxdWlyZXNfc2FmZXR5X2NoZWNrZXI9RmFsc2UKICAgICkKICAg"
        "IHBpcGUgPSBwaXBlLnRvKGRldmljZSkKICAgIHBpcGUuc2V0X3Byb2dyZXNzX2Jhcl9jb25maWcoZGlzYWJsZT1UcnVlKQogICAgaWYgZGV2aWNlID09ICJj"
        "dWRhIjoKICAgICAgICBwaXBlLmVuYWJsZV9hdHRlbnRpb25fc2xpY2luZygpICAgIyBsb3dlciBwZWFrIFZSQU0sIHNtYWxsIHNwZWVkIGNvc3QKICAgIHJl"
        "dHVybiB0b3JjaCwgcGlwZSwgZGV2aWNlCgoKZGVmIGRlY29kZV9sYXRlbnQocGlwZSwgbGF0ZW50cywgdG9yY2gpOgogICAgIiIiCiAgICBUdXJuIGEgbGF0"
        "ZW50IGludG8gYSB2aWV3YWJsZSBpbWFnZS4KCiAgICBUaGlzIGlzIHRoZSBWQUUgZGVjb2RlciBmcm9tIENoYXB0ZXIgMyDigJQgdGhlIHNlY29uZCBoYWxm"
        "IG9mIHRoZQogICAgZW5jb2Rlci9kZWNvZGVyIHBhaXIuIEV2ZXJ5ICdwcm9ncmVzcycgZnJhbWUgeW91IGhhdmUgZXZlciBzZWVuIGluIGFuCiAgICBpbWFn"
        "ZSBnZW5lcmF0b3IgaXMgYSBsYXRlbnQgcHVzaGVkIHRocm91Z2ggdGhpcy4KICAgICIiIgogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgc2Nh"
        "bGVkID0gbGF0ZW50cyAvIHBpcGUudmFlLmNvbmZpZy5zY2FsaW5nX2ZhY3RvcgogICAgICAgIGltYWdlID0gcGlwZS52YWUuZGVjb2RlKHNjYWxlZC50byhw"
        "aXBlLnZhZS5kdHlwZSkpLnNhbXBsZQogICAgaW1hZ2UgPSAoaW1hZ2UgLyAyICsgMC41KS5jbGFtcCgwLCAxKQogICAgaW1hZ2UgPSBpbWFnZS5jcHUoKS5w"
        "ZXJtdXRlKDAsIDIsIDMsIDEpLmZsb2F0KCkubnVtcHkoKQogICAgcmV0dXJuIHBpcGUubnVtcHlfdG9fcGlsKGltYWdlKVswXQoKCmRlZiBnZW5lcmF0ZV93"
        "aXRoX3RyYWNlKHRvcmNoLCBwaXBlLCBwcm9tcHQsIHN0ZXBzLCBndWlkYW5jZSwgc2VlZCwgb3V0ZGlyKToKICAgICIiIkdlbmVyYXRlIG9uZSBpbWFnZSwg"
        "c2F2aW5nIGEgUE5HIGF0IGV2ZXJ5IGRlbm9pc2luZyBzdGVwLiIiIgogICAgb3MubWFrZWRpcnMob3V0ZGlyLCBleGlzdF9vaz1UcnVlKQogICAgZ2VuZXJh"
        "dG9yID0gdG9yY2guR2VuZXJhdG9yKGRldmljZT0iY3B1IikubWFudWFsX3NlZWQoc2VlZCkKICAgIHNhdmVkID0gW10KCiAgICBkZWYgb25fc3RlcChwaXBl"
        "X3JlZiwgc3RlcCwgdGltZXN0ZXAsIGt3YXJncyk6CiAgICAgICAgIyBOb3QgZXZlcnkgc2NoZWR1bGVyIHJ1bnMgZXhhY3RseSBudW1faW5mZXJlbmNlX3N0"
        "ZXBzIGNhbGxiYWNrcy4gU0QKICAgICAgICAjIDEuNSdzIGRlZmF1bHQgUE5ETSBzY2hlZHVsZXIgcnVucyBvbmUgZXh0cmEsIHNvIHRydXN0IHRoZSBzY2hl"
        "ZHVsZXIncwogICAgICAgICMgb3duIHRpbWVzdGVwIGNvdW50IHJhdGhlciB0aGFuIHdoYXQgd2UgYXNrZWQgZm9yIOKAlCBvdGhlcndpc2UgdGhlCiAgICAg"
        "ICAgIyBwcm9ncmVzcyBsaW5lIHJlYWRzICJzdGVwIDcvNiIuCiAgICAgICAgdG90YWwgPSBsZW4oZ2V0YXR0cihwaXBlX3JlZi5zY2hlZHVsZXIsICJ0aW1l"
        "c3RlcHMiLCBbXSkpIG9yIHN0ZXBzCiAgICAgICAgbGF0ZW50cyA9IGt3YXJnc1sibGF0ZW50cyJdCiAgICAgICAgaW1nID0gZGVjb2RlX2xhdGVudChwaXBl"
        "X3JlZiwgbGF0ZW50cywgdG9yY2gpCiAgICAgICAgcGF0aCA9IG9zLnBhdGguam9pbihvdXRkaXIsIGYic3RlcF97c3RlcCArIDE6MDNkfS5wbmciKQogICAg"
        "ICAgIGltZy5zYXZlKHBhdGgpCiAgICAgICAgc2F2ZWQuYXBwZW5kKHBhdGgpCiAgICAgICAgcHJpbnQoZiIgIHN0ZXAge3N0ZXAgKyAxOj4zfS97dG90YWx9"
        "ICAtPiAge29zLnBhdGguYmFzZW5hbWUocGF0aCl9IikKICAgICAgICByZXR1cm4ga3dhcmdzCgogICAgcHJpbnQoZidQcm9tcHQ6ICJ7cHJvbXB0fSInKQog"
        "ICAgcHJpbnQoZiJTdGVwczoge3N0ZXBzfSAgIEd1aWRhbmNlOiB7Z3VpZGFuY2V9ICAgU2VlZDoge3NlZWR9XG4iKQogICAgcHJpbnQoIlNhdmluZyBldmVy"
        "eSBzdGVwLiBXYXRjaCB0aGUgZm9nIHJlc29sdmUuXG4iKQoKICAgIHJlc3VsdCA9IHBpcGUoCiAgICAgICAgcHJvbXB0PXByb21wdCwKICAgICAgICBudW1f"
        "aW5mZXJlbmNlX3N0ZXBzPXN0ZXBzLAogICAgICAgIGd1aWRhbmNlX3NjYWxlPWd1aWRhbmNlLAogICAgICAgIGdlbmVyYXRvcj1nZW5lcmF0b3IsCiAgICAg"
        "ICAgY2FsbGJhY2tfb25fc3RlcF9lbmQ9b25fc3RlcCwKICAgICAgICBjYWxsYmFja19vbl9zdGVwX2VuZF90ZW5zb3JfaW5wdXRzPVsibGF0ZW50cyJdLAog"
        "ICAgKQogICAgZmluYWwgPSBvcy5wYXRoLmpvaW4ob3V0ZGlyLCAiZmluYWwucG5nIikKICAgIHJlc3VsdC5pbWFnZXNbMF0uc2F2ZShmaW5hbCkKICAgIHBy"
        "aW50KGYiXG4gIGZpbmFsICAgICAgICAgIC0+ICB7ZmluYWx9IikKICAgIHJldHVybiBzYXZlZAoKCmRlZiBzd2VlcF9zdGVwcyh0b3JjaCwgcGlwZSwgcHJv"
        "bXB0LCBndWlkYW5jZSwgc2VlZCwgb3V0ZGlyKToKICAgICIiIlNhbWUgc2VlZCBhbmQgcHJvbXB0IGF0IGluY3JlYXNpbmcgc3RlcCBjb3VudHMuIEZpbmQg"
        "dGhlIHBsYXRlYXUuIiIiCiAgICBvcy5tYWtlZGlycyhvdXRkaXIsIGV4aXN0X29rPVRydWUpCiAgICBwcmludCgiU3RlcC1jb3VudCBzd2VlcDogd2hlcmUg"
        "ZG9lcyBpdCBzdG9wIGltcHJvdmluZz9cbiIpCiAgICBmb3IgbiBpbiAoMiwgNSwgMTAsIDIwLCAzNSwgNTApOgogICAgICAgIGcgPSB0b3JjaC5HZW5lcmF0"
        "b3IoZGV2aWNlPSJjcHUiKS5tYW51YWxfc2VlZChzZWVkKQogICAgICAgIGltZyA9IHBpcGUocHJvbXB0PXByb21wdCwgbnVtX2luZmVyZW5jZV9zdGVwcz1u"
        "LAogICAgICAgICAgICAgICAgICAgZ3VpZGFuY2Vfc2NhbGU9Z3VpZGFuY2UsIGdlbmVyYXRvcj1nKS5pbWFnZXNbMF0KICAgICAgICBwYXRoID0gb3MucGF0"
        "aC5qb2luKG91dGRpciwgZiJzdGVwc197bjowMmR9LnBuZyIpCiAgICAgICAgaW1nLnNhdmUocGF0aCkKICAgICAgICBwcmludChmIiAge246PjJ9IHN0ZXBz"
        "IC0+IHtvcy5wYXRoLmJhc2VuYW1lKHBhdGgpfSIpCiAgICBwcmludCgiXG4gIExpbmUgdGhlbSB1cC4gU29tZXdoZXJlIGJldHdlZW4gMjAgYW5kIDM1IHRo"
        "ZSByZXR1cm5zIHN0b3AuIikKICAgIHByaW50KCIgIFRoYXQgcG9pbnQgaXMgeW91ciBxdWFsaXR5L2Nvc3QgZnJvbnRpZXIsIG1lYXN1cmVkIHJhdGhlciB0"
        "aGFuIikKICAgIHByaW50KCIgIGd1ZXNzZWQg4oCUIGFuZCBpdCBpcyB0aGUgc2FtZSBzaGFwZSBvZiBjdXJ2ZSB5b3Ugd2lsbCBtZWV0IGFnYWluIikKICAg"
        "IHByaW50KCIgIGluIExhYiBGLCB3aGVyZSB0aGUgYXhpcyBpcyB0cmFpbmluZyBleGFtcGxlcyBpbnN0ZWFkIG9mIHN0ZXBzLiIpCgoKZGVmIHN3ZWVwX2d1"
        "aWRhbmNlKHRvcmNoLCBwaXBlLCBwcm9tcHQsIHN0ZXBzLCBzZWVkLCBvdXRkaXIpOgogICAgIiIiQ0ZHIHNjYWxlIGlzIGEgdmVjdG9yIG11bHRpcGxpY2F0"
        "aW9uLiBIZXJlIGlzIHdoYXQgaXQgbG9va3MgbGlrZS4iIiIKICAgIG9zLm1ha2VkaXJzKG91dGRpciwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHByaW50KCJHdWlk"
        "YW5jZSBzd2VlcDogd2F0Y2ggaXQgZ28gZnJvbSBpZ25vcmluZyB5b3UgdG8gb3ZlcmNvb2tpbmcuXG4iKQogICAgZm9yIGdfc2NhbGUgaW4gKDEuMCwgMy4w"
        "LCA3LjUsIDE1LjAsIDI1LjApOgogICAgICAgIGcgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPSJjcHUiKS5tYW51YWxfc2VlZChzZWVkKQogICAgICAgIGlt"
        "ZyA9IHBpcGUocHJvbXB0PXByb21wdCwgbnVtX2luZmVyZW5jZV9zdGVwcz1zdGVwcywKICAgICAgICAgICAgICAgICAgIGd1aWRhbmNlX3NjYWxlPWdfc2Nh"
        "bGUsIGdlbmVyYXRvcj1nKS5pbWFnZXNbMF0KICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKG91dGRpciwgZiJndWlkYW5jZV97Z19zY2FsZTowNC4xZn0u"
        "cG5nIikKICAgICAgICBpbWcuc2F2ZShwYXRoKQogICAgICAgIHByaW50KGYiICBjZmcge2dfc2NhbGU6PjQuMWZ9IC0+IHtvcy5wYXRoLmJhc2VuYW1lKHBh"
        "dGgpfSIpCiAgICBwcmludCgiXG4gIEF0IDEuMCB0aGUgbW9kZWwgd2FuZGVycyB3aGVyZXZlciBpdCBsaWtlcyDigJQgeW91ciB3b3JkcyBiYXJlbHkiKQog"
        "ICAgcHJpbnQoIiAgcmVnaXN0ZXIuIEF0IDcuNSB5b3UgZ2V0IGFkaGVyZW5jZSB3aXRoIG5hdHVyYWwgcmVzdWx0cy4gQXQgMjUiKQogICAgcHJpbnQoIiAg"
        "aXQgaXMgY29udG9ydGVkIGFuZCBvdmVyc2F0dXJhdGVkLCBiZWNhdXNlIHlvdSBhbXBsaWZpZWQgdGhlIikKICAgIHByaW50KCIgIHRleHQgZGlyZWN0aW9u"
        "IHNvIGhhcmQgdGhhdCB5b3UgcHVzaGVkIHRoZSBsYXRlbnQgb3V0c2lkZSB0aGUiKQogICAgcHJpbnQoIiAgcmVnaW9uIHdoZXJlIHRoZSBkZWNvZGVyIHBy"
        "b2R1Y2VzIGFueXRoaW5nIHNhbmUuIikKICAgIHByaW50KCJcbiAgWW91IGFyZSB3YXRjaGluZyBhIHZlY3RvciBnZXQgbXVsdGlwbGllZC4iKQoKCmRlZiBz"
        "d2VlcF9zZWVkcyh0b3JjaCwgcGlwZSwgcHJvbXB0LCBzdGVwcywgZ3VpZGFuY2UsIG91dGRpciwgbj02KToKICAgICIiIkZpeGVkIHByb21wdCwgZGlmZmVy"
        "ZW50IHN0YXJ0aW5nIG5vaXNlLiIiIgogICAgb3MubWFrZWRpcnMob3V0ZGlyLCBleGlzdF9vaz1UcnVlKQogICAgcHJpbnQoIlNlZWQgc3dlZXA6IHNhbWUg"
        "d29yZHMsIGRpZmZlcmVudCBzdGFydGluZyBwb2ludC5cbiIpCiAgICBmb3IgcyBpbiByYW5nZShuKToKICAgICAgICBnID0gdG9yY2guR2VuZXJhdG9yKGRl"
        "dmljZT0iY3B1IikubWFudWFsX3NlZWQoMTAwMCArIHMpCiAgICAgICAgaW1nID0gcGlwZShwcm9tcHQ9cHJvbXB0LCBudW1faW5mZXJlbmNlX3N0ZXBzPXN0"
        "ZXBzLAogICAgICAgICAgICAgICAgICAgZ3VpZGFuY2Vfc2NhbGU9Z3VpZGFuY2UsIGdlbmVyYXRvcj1nKS5pbWFnZXNbMF0KICAgICAgICBwYXRoID0gb3Mu"
        "cGF0aC5qb2luKG91dGRpciwgZiJzZWVkX3sxMDAwICsgc30ucG5nIikKICAgICAgICBpbWcuc2F2ZShwYXRoKQogICAgICAgIHByaW50KGYiICBzZWVkIHsx"
        "MDAwICsgc30gLT4ge29zLnBhdGguYmFzZW5hbWUocGF0aCl9IikKICAgIHByaW50KCJcbiAgRXZlcnkgb25lIG9mIHRob3NlIGdvdCBpZGVudGljYWwgaW5z"
        "dHJ1Y3Rpb25zLiBUaGUgdmFyaWF0aW9uIikKICAgIHByaW50KCIgIGlzIGVudGlyZWx5IGluIHdoZXJlIHRoZSB3YWxrIHN0YXJ0ZWQuIikKICAgIHByaW50"
        "KCkKICAgIHByaW50KCIgIFRoaXMgZGlzdGluY3Rpb24gbWF0dGVycyBpbiBwcmFjdGljZTogJ3RoZSBwcm9tcHQgd2FzIGFtYmlndW91cyciKQogICAgcHJp"
        "bnQoIiAgYW5kICd0aGUgd2FsayBzdGFydGVkIHNvbWV3aGVyZSBlbHNlJyBhcmUgZGlmZmVyZW50IGRpYWdub3NlcyIpCiAgICBwcmludCgiICB3aXRoIGRp"
        "ZmZlcmVudCBmaXhlcywgYW5kIHBlb3BsZSBjb25mdXNlIHRoZW0gY29uc3RhbnRseS4iKQoKCmRlZiBtYWluKCk6CiAgICBwID0gYXJncGFyc2UuQXJndW1l"
        "bnRQYXJzZXIoZGVzY3JpcHRpb249IkxhYiBDOiB3YXRjaCB0aGUgZGVub2lzaW5nIHdhbGsuIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW1vZGVsIiwgZGVm"
        "YXVsdD0ic3RhYmxlLWRpZmZ1c2lvbi12MS01L3N0YWJsZS1kaWZmdXNpb24tdjEtNSIsCiAgICAgICAgICAgICAgICAgICBoZWxwPSJzZWUgLi4vQ1VSUkVO"
        "VC5tZCIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1wcm9tcHQiLCBkZWZhdWx0PURFRkFVTFRfUFJPTVBUKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc3RlcHMi"
        "LCB0eXBlPWludCwgZGVmYXVsdD0yNSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWd1aWRhbmNlIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD03LjUpCiAgICBwLmFk"
        "ZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCBkZWZhdWx0PSJvdXRwdXQiKQog"
        "ICAgcC5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD1Ob25lLCBjaG9pY2VzPVtOb25lLCAiY3VkYSIsICJtcHMiLCAiY3B1Il0pCiAgICBwLmFk"
        "ZF9hcmd1bWVudCgiLS1zd2VlcC1zdGVwcyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1zd2VlcC1ndWlkYW5jZSIsIGFj"
        "dGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1zd2VlcC1zZWVkcyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzID0gcC5w"
        "YXJzZV9hcmdzKCkKCiAgICB0b3JjaCwgcGlwZSwgZGV2aWNlID0gbG9hZF9waXBlbGluZShhcmdzLm1vZGVsLCBhcmdzLmRldmljZSkKCiAgICBpZiBhcmdz"
        "LnN3ZWVwX3N0ZXBzOgogICAgICAgIHN3ZWVwX3N0ZXBzKHRvcmNoLCBwaXBlLCBhcmdzLnByb21wdCwgYXJncy5ndWlkYW5jZSwgYXJncy5zZWVkLAogICAg"
        "ICAgICAgICAgICAgICAgIG9zLnBhdGguam9pbihhcmdzLm91dCwgInN3ZWVwX3N0ZXBzIikpCiAgICBlbGlmIGFyZ3Muc3dlZXBfZ3VpZGFuY2U6CiAgICAg"
        "ICAgc3dlZXBfZ3VpZGFuY2UodG9yY2gsIHBpcGUsIGFyZ3MucHJvbXB0LCBhcmdzLnN0ZXBzLCBhcmdzLnNlZWQsCiAgICAgICAgICAgICAgICAgICAgICAg"
        "b3MucGF0aC5qb2luKGFyZ3Mub3V0LCAic3dlZXBfZ3VpZGFuY2UiKSkKICAgIGVsaWYgYXJncy5zd2VlcF9zZWVkczoKICAgICAgICBzd2VlcF9zZWVkcyh0"
        "b3JjaCwgcGlwZSwgYXJncy5wcm9tcHQsIGFyZ3Muc3RlcHMsIGFyZ3MuZ3VpZGFuY2UsCiAgICAgICAgICAgICAgICAgICAgb3MucGF0aC5qb2luKGFyZ3Mu"
        "b3V0LCAic3dlZXBfc2VlZHMiKSkKICAgIGVsc2U6CiAgICAgICAgZ2VuZXJhdGVfd2l0aF90cmFjZSh0b3JjaCwgcGlwZSwgYXJncy5wcm9tcHQsIGFyZ3Mu"
        "c3RlcHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBhcmdzLmd1aWRhbmNlLCBhcmdzLnNlZWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBv"
        "cy5wYXRoLmpvaW4oYXJncy5vdXQsICJ0cmFjZSIpKQogICAgICAgIHByaW50KCkKICAgICAgICBwcmludCgiPSIgKiA3MCkKICAgICAgICBwcmludCgiTk9X"
        "IExPT0sgQVQgVEhFTSBJTiBPUkRFUiIpCiAgICAgICAgcHJpbnQoIj0iICogNzApCiAgICAgICAgcHJpbnQoZiIgIE9wZW4ge29zLnBhdGguam9pbihhcmdz"
        "Lm91dCwgJ3RyYWNlJyl9IGFuZCBzY2FuIHN0ZXBfMDAxIG9ud2FyZC4iKQogICAgICAgIHByaW50KCkKICAgICAgICBwcmludCgiICBUaGUgdGhpbmcgbm9i"
        "b2R5IHRlbGxzIHlvdSBpbiBhZHZhbmNlOiBub3RpY2UgSE9XIEVBUkxZIHRoZSIpCiAgICAgICAgcHJpbnQoIiAgY29tcG9zaXRpb24gaXMgZml4ZWQuIFdp"
        "dGhpbiB0aGUgZmlyc3QgZmlmdGggb2YgdGhlIHN0ZXBzIOKAlCIpCiAgICAgICAgcHJpbnQoIiAgYmVmb3JlIHRoZXJlIGlzIGFueXRoaW5nIHlvdSB3b3Vs"
        "ZCBjYWxsIGRldGFpbCDigJQgdGhlIG92ZXJhbGwiKQogICAgICAgIHByaW50KCIgIGxheW91dCBpcyBhbHJlYWR5IGNvbW1pdHRlZC4gRXZlcnl0aGluZyBh"
        "ZnRlciB0aGF0IGlzIikKICAgICAgICBwcmludCgiICByZWZpbmVtZW50IGluc2lkZSBhIGRlY2lzaW9uIHRoYXQgaGFzIGFscmVhZHkgYmVlbiBtYWRlLiIp"
        "CiAgICAgICAgcHJpbnQoKQogICAgICAgIHByaW50KCIgIFRoZW4gcnVuIHRoZSB0aHJlZSBzd2VlcHM6IikKICAgICAgICBwcmludCgiICAgIHB5dGhvbiBk"
        "ZW5vaXNlX2luX3B1YmxpYy5weSAtLXN3ZWVwLXN0ZXBzIikKICAgICAgICBwcmludCgiICAgIHB5dGhvbiBkZW5vaXNlX2luX3B1YmxpYy5weSAtLXN3ZWVw"
        "LWd1aWRhbmNlIikKICAgICAgICBwcmludCgiICAgIHB5dGhvbiBkZW5vaXNlX2luX3B1YmxpYy5weSAtLXN3ZWVwLXNlZWRzIikKICAgICAgICBwcmludCgp"
        "CgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="
    ),
}

for name, blob in FILES.items():
    pathlib.Path(name).write_bytes(base64.b64decode(blob))
    print('wrote', name)


In [ ]:
!python denoise_in_public.py


## Lab C — save every denoising step

In [ ]:
!python denoise_in_public.py --steps 25

Display them in order. **Notice how early the composition is fixed** — within the first fifth of the steps, before there's any detail.

In [ ]:
import glob
from IPython.display import display
from PIL import Image
paths = sorted(glob.glob('output/trace/step_*.png'))
for p in paths[::max(1, len(paths)//8)]:
    print(p)
    display(Image.open(p).resize((256, 256)))

## The three sweeps

In [ ]:
!python denoise_in_public.py --sweep-steps

In [ ]:
!python denoise_in_public.py --sweep-guidance

In [ ]:
!python denoise_in_public.py --sweep-seeds

## Lab D — the interpolation walk

Ten images along the straight line between two prompt embeddings, same seed
throughout.

In [ ]:
%cd /content/latent-space-labs/lab-d-interpolation
!python interpolate.py --frames 10

In [ ]:
import glob
from IPython.display import display
from PIL import Image
for p in sorted(glob.glob('output/walk/frame_*.png')):
    print(p)
    display(Image.open(p).resize((256, 256)))

**It is a morph, not a crossfade.** Frame 5 is a coherent image of one
thing halfway between two concepts — not two images overlaid.

If the model were a lookup table of memorised images, the midpoints would be
garbage or would snap between the endpoints. They don't. The space is
continuous, which means the model is navigating, not retrieving.

Every middle frame is an image of something that has no name.

## Close the loop

Now **recognise** what you just **generated**, with the same CLIP machinery.

In [ ]:
!python interpolate.py --frames 10 --classify

The middle frames come back near-equal on both prompts, flagged
ambiguous, because that is genuinely where they sit.

Generation picked an address and built what belongs there. Recognition read the
address something arrived at. **Same map, both directions.**